# GBDT/Treesのバイアス

| 論点                      |              何が偏るか | 原因                         | 代表的な対策                                                       |
| ----------------------- | -----------------: | -------------------------- | ------------------------------------------------------------ |
| split selection bias    |            木が選ぶ特徴量 | 候補分割数が多い特徴ほど偶然よい split が出る | conditional inference trees, unbiased split selection, OOB評価 |
| gain importance bias    |             特徴量重要度 | 訓練データ上の impurity/gain を足す  | permutation importance, OOB gain, unbiased gain              |
| prediction shift        |        boostingの予測 | 自分自身の目的変数情報が学習過程に混入        | CatBoost ordered boosting                                    |
| high-cardinality bias   |   連続値・多カテゴリ特徴の過大評価 | 分割候補数・自由度が大きい              | カテゴリ処理、正則化、OOB/validation評価                                  |
| correlated feature bias | 相関特徴の重要度分散・過小/過大評価 | 片方で代替できる                   | grouped permutation importance, feature clustering           |


## Gain/Feature Importanceのバイアス

### 過学習によるGain推定のバイアス

不純度は過学習を起こしている場合、重要でない特徴量に高い重要度を与えてしまうことがある（Permutation Importanceはこの問題を持たない）（[scikit-learn documentation](https://scikit-learn.org/stable/modules/permutation_importance.html#relation-to-impurity-based-importance-in-trees)）

標準的なGain（情報利得）の推定では、目的変数と独立な特徴量のGainでもゼロになることはめったになく、多くの場合でプラスの値になる（→ 重要じゃない特徴量でもGainはプラスになりがち）。
また、この問題はGain推定時にtraining setだけじゃなくvalidation setも使うことで解消する（[Zhang et al., 2023](https://dl.acm.org/doi/abs/10.24963/ijcai.2023/515)）

### 高カーディナリティ特徴量を高く評価するバイアス

二値変数やカテゴリ数が少ないカテゴリ変数などの低カーディナリティの特徴よりも、高カーディナリティの特徴（典型的には連続変数）のGainを高く評価し、分岐候補点の探索でも高カーディナリティ特徴で分岐しがちな傾向がある（[scikit-learn documentation](https://scikit-learn.org/stable/modules/permutation_importance.html#relation-to-impurity-based-importance-in-trees); [Zhang et al., 2023](https://dl.acm.org/doi/abs/10.24963/ijcai.2023/515); [Hothorn et al., 2006](https://doi.org/10.1198/106186006X133933)）




## prediction shift

CatBoostの「unbiased boosting」で対象にしているのはprediction shift、つまり各サンプルの予測を作るときにそのサンプル自身の目的変数情報が過去の木や target statistics に混入する一種の target leakage 。



## 参考

- Hothorn, T., Hornik, K., & Zeileis, A. (2006). Unbiased recursive partitioning: A conditional inference framework. Journal of Computational and Graphical statistics, 15(3), 651-674. https://doi.org/10.1198/106186006X133933